# 04 -- Filtered vs unfiltered

Compare the frequency content of raw vs filtered data.

In [1]:
import sys
sys.path.insert(0, "../src")            # run from the examples/ folder

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.00 GB / avail 0.51 GB (98%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

## Build raw and filtered handles

In [3]:
raw = ds.MOF00135.window(WSTART, WLEN).baseline()
b1  = raw.filter(0.1, 24.9)
b2  = raw.filter(1.0, 10.0)

## Fourier of each

In [4]:
f_raw = raw.fourier(component="x")
f_b1  = b1.fourier(component="x")
f_b2  = b2.fourier(component="x")
print("dominant freqs raw :", [round(float(x),2) for x in f_raw["dom_freqs"]])
print("dominant freqs 1-10:", [round(float(x),2) for x in f_b2["dom_freqs"]])

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)


[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)
dominant freqs raw : [46.03, 14.51, 9.46, 17.77]
dominant freqs 1-10: [5.82, 3.03, 8.87]


## Overlay the spectra

In [5]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9,3.5))
plt.semilogy(f_raw["freqs"], f_raw["spectrum"] + 1e-20, label="raw")
plt.semilogy(f_b1["freqs"],  f_b1["spectrum"]  + 1e-20, label="0.1-24.9 Hz")
plt.semilogy(f_b2["freqs"],  f_b2["spectrum"]  + 1e-20, label="1-10 Hz")
plt.xlim(0, 30); plt.xlabel("Frequency [Hz]"); plt.ylabel("FFT power")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\ppala\AppData\Local\Temp\claude\ipykernel_840224\2528161849.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## PSD band energy: raw vs filtered

In [6]:
print("raw  bands:", {str(k): round(float(v),6) for k,v in raw.psd(component="x")["band_energy"].items()})
print("1-10 bands:", {str(k): round(float(v),6) for k,v in b2.psd(component="x")["band_energy"].items()})

[psd] MOF00135 comp=x nperseg=256 (computed)
raw  bands: {'(0, 1)': 0.0, '(1, 2.5)': 0.0, '(2.5, 5)': 0.0, '(5, 10)': 0.0}
[psd] MOF00135 comp=x nperseg=256 (computed)
1-10 bands: {'(0, 1)': 0.0, '(1, 2.5)': 0.0, '(2.5, 5)': 0.0, '(5, 10)': 0.0}
